# VESTIGE — masking-strategy benchmark (GPU)

Fine-tunes DNABERT-2 under all 8 collator configs on a Colab T4. ~1-2 min/run.

**First:** Runtime -> Change runtime type -> **T4 GPU** -> Save. Then run top to bottom.

In [ ]:
# 1. confirm a GPU is attached
!nvidia-smi -L
import torch; print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())

In [ ]:
# 2. upload vestige_train.zip (made on your Mac), then unzip
from google.colab import files
import zipfile, os
up = files.upload()                      # pick vestige_train.zip
zname = next(iter(up))
with zipfile.ZipFile(zname) as z: z.extractall('vestige')
os.chdir('vestige')
print('cwd:', os.getcwd()); print('contents:', sorted(os.listdir('.')))

In [ ]:
# 3. deps. REMOVE triton: DNABERT-2's triton attention kernels crash on Colab's modern triton;
#    without it the model uses PyTorch attention (still GPU-fast on CUDA).
!pip install -q 'transformers==4.57.1' 'datasets==4.5.0' einops accelerate
!pip uninstall -y triton pytorch-triton 2>/dev/null
print('deps ready, triton removed')

In [ ]:
# 4. train all 8 configs; surface errors and stop at the first failure
import subprocess, os, time
os.environ['USE_TF']='0'; os.environ['USE_FLAX']='0'
configs = ['config_mlm','config_dam','config_span','config_pmdmask',
           'config_dam_noscale','config_dam_atmask','config_dam_smooth','config_dam_power']
ok = []
for c in configs:
    print(f"\n{'='*56}\n  {c}\n{'='*56}", flush=True); t = time.time()
    r = subprocess.run(['python','training/train.py',f'training/{c}.yaml'], capture_output=True, text=True)
    if r.returncode == 0:
        ok.append(c); print(f'>>> {c} OK in {(time.time()-t)/60:.1f} min', flush=True)
    else:
        print(f'>>> {c} FAILED exit={r.returncode}\n{r.stderr[-1800:]}', flush=True); break
print(f'\nSUCCEEDED {len(ok)}/8:', ok)

In [ ]:
# 5. keep best model per run, zip, save to Drive
import os, shutil, glob
for run in glob.glob('training/checkpoints/*'):
    for ck in glob.glob(os.path.join(run,'checkpoint-*')): shutil.rmtree(ck, ignore_errors=True)
shutil.make_archive('vestige_checkpoints','zip','training/checkpoints')
print('zip MB:', round(os.path.getsize('vestige_checkpoints.zip')/1e6,1))
from google.colab import drive; drive.mount('/content/drive')
shutil.copy('vestige_checkpoints.zip','/content/drive/MyDrive/vestige_checkpoints.zip')
print('saved to Drive: MyDrive/vestige_checkpoints.zip')
# or: from google.colab import files; files.download('vestige_checkpoints.zip')